# Cramér–Rao bounds

In [ ]:
import jax.numpy as jnp
import numpy as np
import jax
from tqdm import tqdm
import matplotlib.pyplot as plt
from master import *

jax.config.update("jax_enable_x64", True)
jax.devices()
device_id = 1
jax.config.update("jax_default_device", jax.devices()[device_id])

In [ ]:
N_crop = 256
N_pad = 128
N_photons = 2.5e4
nonzero_cameraman = cameraman() * 0.9 + 0.1
x_cameraman = nonzero_cameraman
x_mandrill = as_phase_field(mandrill() ** 2 - 0.5)
x_obj, obj_support = crop_and_pad(x_cameraman * x_mandrill, N_crop=N_crop, N_pad=N_pad)
x_obj = normalize_image_intensity(x_obj)

x, y, X, Y, R, k_0, k_x, k_y, K_X, K_Y, k, r_extent, k_extent = create_physical_axes(
    x_obj
)

use_cos_filter = False
filter_I = True

if use_cos_filter:
    i = np.arange(0, N_crop, dtype=np.float64)
    j = np.arange(0, N_crop, dtype=np.float64)
    I, J = np.meshgrid(i, j)
    filt = np.cos(jnp.pi / 2 * (I / (N_crop / 2) - 1)) * np.cos(
        jnp.pi / 2 * (J / (N_crop / 2) - 1)
    )
    if filter_I:
        filt = jnp.sqrt(filt)

    # x_obj = jnp.ones_like(x_obj)
    x_obj, _ = crop_and_pad(
        center_crop(x_obj, N_crop) * filt,
        N_crop=N_crop,
        N_pad=N_pad,
    )
x_obj = normalize_image_intensity(x_obj)
plot_intensity_phase(x_obj, r_extent, label="y", vertical=True)
K = x_obj.size
K

In [ ]:
F_pupil = jnp.ones_like(k)

spectral_oversampling = analyse_oversampling(F_pupil)
obj_oversampling = analyse_oversampling(obj_support)
print(
    f"Spectral oversampling: {spectral_oversampling};  Object oversampling: {obj_oversampling}"
)
y = ifft2_shifted(x_obj)
plot_intensity_phase(y, k_extent, label=r"\hat{x'}", use_log=True)

In [ ]:
x = fft2_shifted(y)
plot_intensity_phase(x, r_extent, label="y")

## 1. Bouchet style bounds on CRB
First some fundamental bounds are considered as described in Fundamental Bounds on the Precision of Classical Phase Microscopes. Assume an object field with 
$$x_k = \sqrt{n_k}e^{i\phi_k}$$
so that $n_k$ photons (on average) are indicent on the associated area of pixel $k$. It can be derived from the diagonal FIM terms that
$$\operatorname{Var} (\hat{n}_k) \geq n_k$$
$$\operatorname{Var} (\hat{\phi}_k) \geq \frac{1}{4n_k}$$

In [ ]:
from master.noise import get_unit_photons

x_ph = get_unit_photons(x, N_photons=N_photons)
y_ph = get_unit_photons(y, N_photons=N_photons)
plot_intensity_phase(x_ph, r_extent, label="y")
plot_intensity_phase(y_ph, r_extent, label="y")

## 1.1. Lower bound on CRB for $n_k$

In [ ]:
n_k = jnp.abs(x_ph) ** 2
# Just poisson statistics
n_k_var_bouchet = n_k
n_k_std_bouchet = jnp.sqrt(n_k)
# plot(n_k_std_bouchet, label=r"$[\sigma_{\hat{n}_k}]^{\text{Bouchet}}$")
plot(
    center_crop(n_k_std_bouchet, N_crop),
    label=r"$[\sigma_{\hat{n}_k}]^{\text{Bouchet}}$",
)

## 1.2. Lower bound on CRB for phase $\phi_k$

In [ ]:
n_k = jnp.abs(x_ph) ** 2
phi_var_bouchet = 1 / (4 * n_k)
phi_std_bouchet = jnp.sqrt(phi_var_bouchet)
# plot(phi_std_bouchet * obj_support, label=r"$[\sigma_{\hat{\phi}_k}]^{Bouchet}$")
plot(center_crop(phi_std_bouchet, N_crop), label=r"$[\sigma_{\hat{\phi}_k}]^{Bouchet}$")

# 2. Bounds from exact diagonal elements


In [ ]:
import jax
import jax.numpy as jnp
from master.estimator_bounds import fisher_2x2_block_fast

Jn_diag, Jphi_diag, Jnphi_off, n_var_diag, phi_var_diag = fisher_diag_bouchet_fast(
    x_ph, y_ph
)

# from numpy.testing import assert_allclose

# assert_allclose(Jn_diag_fast, Jn_diag, rtol=1e-12, atol=1e-12)
# assert_allclose(Jphi_diag_fast, Jphi_diag, rtol=1e-12, atol=1e-12)
# assert_allclose(Jnphi_off_fast, Jnphi_off, rtol=1e-12, atol=1e-12)

In [ ]:
det_j = Jn_diag * Jphi_diag - Jnphi_off**2

In [ ]:
np.mean(Jn_diag * Jphi_diag), np.mean(Jnphi_off**2)

In [ ]:
n_k_std_num_diag = jnp.sqrt(n_var_diag) * obj_support
phi_k_std_num_diag = jnp.sqrt(phi_var_diag) * obj_support
plot(n_k_std_num_diag, label=r"$[\sigma_{\hat{n}_k}]^{Diagonal CRB}$")
plot(
    phi_k_std_num_diag,
    label=r"$[\sigma_{\hat{\phi}_k}]^{Diagonal CRB}$",
)
plot(
    det_j * obj_support,
    label=r"$[det(J_{jj})]$",
)
# Verify that the difference between this and the optimistic bound is about sqrt(2) with some minor local variation signal
# depending on the object imaged
# plot(
#     jnp.sqrt(n_var_diag) / n_k_std_bouchet * obj_support,
#     label=r"$[\sigma_{\hat{n}_k}]^{Diagonal CRB}$",
# )
# plot(
#     jnp.sqrt(phi_var_diag) / phi_std_bouchet * obj_support,
#     label=r"$[\sigma_{\hat{\phi}_k}]^{Diagonal CRB}$",
# )

# 3. Exact numerical bounds from full $J$ using inverse

In [ ]:
import jax
import jax.numpy as jnp


def centered_ifft2(x):
    return jnp.fft.fftshift(jnp.fft.ifft2(jnp.fft.ifftshift(x), norm="ortho"))


@jax.jit
def fisher_full_bouchet_fft(x_ph, y_ph, eps=1e-12):
    """Note: this requires rather small objects to work. The full N_crop=256, N_pad=128 object requires 1.75TiB of memory."""
    N1, N2 = x_ph.shape
    d = N1 * N2
    P = d

    n = jnp.abs(x_ph) ** 2
    phi = jnp.angle(x_ph)
    psi = jnp.angle(y_ph)

    # One FFT gives the kernel used in all pairwise entries
    g = jnp.exp(-2j * psi)
    H = centered_ifft2(g)  # H[J] = (1/sqrt(d)) sum_k g_k exp(+2πi k·J/N)

    # centered pixel coordinates
    J1 = jnp.arange(N1) - N1 // 2
    J2 = jnp.arange(N2) - N2 // 2
    J1g, J2g = jnp.meshgrid(J1, J2, indexing="ij")

    # flatten pixel data
    j1 = J1g.reshape(-1)  # centered coords of each pixel
    j2 = J2g.reshape(-1)
    phi_flat = phi.reshape(-1)
    n_flat = n.reshape(-1)

    # all pairwise centered sums
    S1 = j1[:, None] + j1[None, :]
    S2 = j2[:, None] + j2[None, :]

    # convert centered summed coords -> array indices
    I1 = (S1 + N1 // 2) % N1
    I2 = (S2 + N2 // 2) % N2

    # Q_ij = sum_k g_k exp(i(beta_{k,i}+beta_{k,j})) = sqrt(d) * H[sum index]
    Q = jnp.sqrt(d) * H[I1, I2]

    # Cplus_ij = exp(i(phi_i + phi_j)) * Q_ij
    phase_sum = jnp.exp(1j * (phi_flat[:, None] + phi_flat[None, :]))
    Cplus = phase_sum * Q

    ReCp = jnp.real(Cplus)
    ImCp = jnp.imag(Cplus)

    sqrt_n = jnp.sqrt(n_flat + eps)
    R = 1.0 / (sqrt_n[:, None] * sqrt_n[None, :])
    S = sqrt_n[:, None] * sqrt_n[None, :]
    T = sqrt_n[None, :] / sqrt_n[:, None]

    I = jnp.eye(P, dtype=ReCp.dtype)

    # full blocks
    Jnn = (R * (d * I + ReCp)) / (2.0 * d)
    Jphiphi = (2.0 / d) * S * (d * I - ReCp)
    Jnphi = -(T * ImCp) / d

    # by symmetry
    Jphin = Jnphi.T

    # assemble full Fisher matrix
    top = jnp.concatenate([Jnn, Jnphi], axis=1)
    bot = jnp.concatenate([Jphin, Jphiphi], axis=1)
    J = jnp.concatenate([top, bot], axis=0)

    return J, Jnn, Jphiphi, Jnphi


J, Jnn, Jphiphi, Jnphi = fisher_full_bouchet_fft(x_ph, y_ph)
J.shape

In [ ]:
support_flat = obj_support.reshape(-1).astype(bool)
idx = jnp.where(support_flat)[0]
P = obj_support.size
idx_n = idx
idx_phi = idx + P
idx_full = jnp.concatenate([idx_n, idx_phi])
J_sup = J[idx_full[:, None], idx_full[None, :]]
J_sup.shape

Use the fact that $J$ is a symmetric matrix that is diagonalizable with real eigenvalues and orthogonal eigenvectors.

In [ ]:
evals, evecs = jnp.linalg.eigh(J_sup)

In [ ]:
rtol = 5e-14
tol = rtol * jnp.max(evals)
evals_pinv = jnp.where(evals > tol, 1.0 / evals, 0.0)
N_support_params = jnp.sum(obj_support * 2)
print(jnp.sum(evals > tol), N_support_params)
J_nn_pinv = (evecs * evals_pinv[None, :]) @ evecs.T

In [ ]:
J_nn_pinv

In [ ]:
def extract_crb_diagonals(J_pinv, shape):
    N1, N2 = shape
    P = N1 * N2

    diag = jnp.diag(J_pinv)

    n_crb = diag[:P].reshape(N1, N2)
    phi_crb = diag[P:].reshape(N1, N2)

    return n_crb, phi_crb


n_num_crb_single_matrix, phi_num_crb_single_matrix = extract_crb_diagonals(
    J_nn_pinv, (N_crop, N_crop)
)
n_num_crb_single_matrix = pad_image(n_num_crb_single_matrix, N_pad)
phi_num_crb_single_matrix = pad_image(phi_num_crb_single_matrix, N_pad)

In [ ]:
plot(jnp.sqrt(n_num_crb_single_matrix), label=r"$[\sigma_{\hat{n}_k}]^{Exact CRB}$")
plot(
    jnp.sqrt(phi_num_crb_single_matrix),
    label=r"$[\sigma_{\hat{\phi}_k}]^{Exact CRB}$",
)

## 3.1. Analyze spectrum of $J$

In [ ]:
for i in range(10):
    i = -i - 1
    P_s = int(jnp.sum(obj_support))
    v = evecs[:, i]  # assuming ascending eigenvalues
    v_n = v[:P_s]
    v_phi = v[P_s:]
    prep_im = lambda v: pad_image(v.reshape((N_crop, N_crop)), pad=N_pad)
    print(f"{i}: ||lambda_v|| = {jnp.linalg.norm(evals[i]):.3g}")
    print(f"     ||v_n|| = {jnp.linalg.norm(v_n):.3g}")
    print(f"     ||v_phi|| = {jnp.linalg.norm(v_phi):.3g}")
    # plot(prep_im(v_n))
    # plot(prep_im(v_phi))
    plot_row([prep_im(v_n), prep_im(v_phi)], labels=[r"v_n", r"v_\phi"])

In [ ]:
plt.hist(jnp.log(evals), bins=64)
plt.show()

In [ ]:
P_s = int(jnp.sum(obj_support))
# v = evecs[:, i]  # assuming ascending eigenvalues
v_n = evecs[:P_s, :]
v_phi = evecs[P_s:, :]
# prep_im = lambda v: pad_image(v.reshape((N_crop, N_crop)), pad=N_pad)

v_n_norm = jnp.sqrt(jnp.sum(jnp.abs(v_n) ** 2, axis=-2))
v_phi_norm = jnp.sqrt(jnp.sum(jnp.abs(v_phi) ** 2, axis=-2))
print(v_n_norm)
print(v_phi_norm)
plt.scatter(v_n_norm, v_phi_norm)
plt.xlabel(r"$|v_k|^2_n$")
plt.ylabel(r"$|v_k|^2_\phi$")
plt.show()

## 3.2. Separately analyze $J_{nn},J_{\phi\phi}$

In [ ]:
# J_nn bit
evals_nn, evecs_nn = jnp.linalg.eigh(J_sup[:P_s, :P_s])
rtol = 5e-12
tol = rtol * jnp.max(evals_nn)
evals_pinv_nn = jnp.where(evals_nn > tol, 1.0 / evals_nn, 0.0)
N_support_params = jnp.sum(obj_support * 2)
print(jnp.sum(evals_nn > tol), P_s)
J_nn_pinv = (evecs_nn * evals_pinv_nn[None, :]) @ evecs_nn.T
plt.hist(jnp.log(evals_nn), bins=32)

n_num_crb = jnp.sqrt(jnp.diag(J_nn_pinv).reshape((N_crop, N_crop)))
n_num_crb = pad_image(n_num_crb, N_pad)
plot(
    n_num_crb,
    label=r"$[\sigma_{\hat{n}_k}]^{Exact CRB}$",
)
# phi_crb = pad_image(phi_crb, N_pad)

In [ ]:
# J_nn bit
evals_phiphi, evecs_phiphi = jnp.linalg.eigh(J_sup[P_s:, P_s:])
rtol = 5e-14
tol = rtol * jnp.max(evals_phiphi)
print(tol, jnp.min(evals_phiphi), jnp.max(evals_phiphi))
eval_zero = evals_phiphi[0]
evec_zero = evecs_phiphi[:, 0]
plot(evec_zero.reshape((N_crop, N_crop)))
evals_phiphi = evals_phiphi[1:]
evecs_phiphi = evecs_phiphi[:, 1:]
evals_phiphi = jnp.where(evals_phiphi > tol, 1.0 / evals_phiphi, 0.0)
N_support_params = jnp.sum(obj_support * 2)
print(jnp.sum(evals_phiphi > tol), P_s)
J_phiphi_pinv = (evecs_phiphi * evals_phiphi[None, :]) @ evecs_phiphi.T
plt.hist(jnp.log(evals_phiphi), bins=32)
phi_num_crb = jnp.sqrt(jnp.diag(J_phiphi_pinv).reshape((N_crop, N_crop)))
phi_num_crb = pad_image(phi_num_crb, N_pad)
plot(
    phi_num_crb,
    label=r"$[\sigma_{\hat{\phi}_k}]^{Exact CRB}$",
)

## 3.3. Verify that $[J^{-1}]_{ii} \geq J_{ii}^{-1}$
This checks that the simple inverse of the exact diagonal elements $J_{ii}$ does not ever give less information than the diagonal element of the full inverse. This is a key property of $J$ as PSD matrix. 

In [ ]:
plot(n_num_crb / n_k_std_num_diag)
plot(phi_num_crb / phi_k_std_num_diag)

# 4. Side to side comparison of estimates and CRB

In [ ]:
import h5py
from stats import estimator_stats

filename = "data/estimates-hio-second-order-ml.h5"
with h5py.File(filename, "r") as f:
    x_est = f["x_est"][:]
    x_true = f["x_true"][:]
    N_crop = f.attrs["N_crop"]
    N_pad = f.attrs["N_pad"]
    N_photons = f.attrs["N_photons"]
    N_samples = f.attrs["N_samples"]

In [ ]:
n_hat = np.abs(x_est) ** 2
n = jnp.abs(x_true) ** 2
n_hat_mu, n_hat_std, n_hat_bias, n_hat_MSE = estimator_stats(n_hat, n)
phi_hat = np.angle(x_est) * obj_support
phi = jnp.angle(x_true) * obj_support
phi_hat_mu, phi_hat_std, phi_hat_bias, phi_hat_MSE = estimator_stats(phi_hat, phi)
plot_row(
    [n_num_crb, n_hat_std, n_hat_bias, n_hat_std / n_num_crb],
    labels=[
        r"[\sigma_{\hat{n}_k}]^{Exact CRB}",
        r"[\sigma_{\hat{n}_k}]^{HIO+ML}",
        r"[\mu_{\hat{n}_k}]^{HIO+ML} - n_k",
        r"\eta(\hat{n}_k)",
    ],
)
plot_row(
    [phi_num_crb, phi_hat_std, phi_hat_bias, phi_hat_std / phi_num_crb],
    labels=[
        r"[\sigma_{\hat{\phi}_k}]^{Exact CRB}",
        r"[\sigma_{\hat{\phi}_k}]^{HIO+ML}",
        r"[\mu_{\hat{\phi}_k}]^{HIO+ML} - \phi_k",
        r"\eta(\hat{\phi}_k)",
    ],
)